In [ ]:
import serial
import time

# --- CONFIGURACIÓN ---
PUERTO_SERIE = 'COM7'      # Cambia según tu puerto
BAUDIOS = 9600
ARCHIVO_PUNTOS = "puntos.txt"

# Correcciones angulares (ajústalas según tu brazo)
CORRECCION_CODO   = 0.0    # positivo aleja
CORRECCION_HOMBRO = -7.0   # positivo aleja
CORRECCION_BASE   = -6.8   # positivo aleja

# Inicializar comunicación serial (ejecutar solo una vez)
arduino = serial.Serial(PUERTO_SERIE, BAUDIOS)
time.sleep(2)   # Esperar a que Arduino reinicie
print("Comunicación serial establecida")

In [ ]:
def cargar_puntos():
    """Carga desde 'puntos.txt' las coordenadas (base, codo, hombro) de cada casilla."""
    puntos = {}
    try:
        with open(ARCHIVO_PUNTOS, "r") as f:
            for linea in f:
                linea = linea.strip()
                if linea == "" or linea.startswith("#"):
                    continue
                partes = linea.split(":")
                if len(partes) == 2:
                    casilla = partes[0].strip().upper()
                    valores = partes[1].strip().split(",")
                    if len(valores) == 3:
                        puntos[casilla] = [float(v) for v in valores]
        print(f"Puntos cargados: {len(puntos)} casillas")
    except FileNotFoundError:
        print(f"ERROR: no se encontró '{ARCHIVO_PUNTOS}'")
    return puntos

In [ ]:
def enviar(valor):
    """Envía un comando al Arduino y espera la respuesta 'done'."""
    arduino.write((str(valor) + "\n").encode())
    print("  Enviado:", valor)
    while True:
        if arduino.in_waiting:
            respuesta = arduino.readline().decode().strip()
            print("  Arduino:", respuesta)
            if respuesta == "done":
                break

In [ ]:
def obtener_punto(casilla):
    """Lee el archivo y devuelve [base, codo, hombro] de la casilla (ej. 'A1')."""
    casilla = casilla.upper()
    puntos = cargar_puntos()
    if casilla in puntos:
        return puntos[casilla]
    print(f"Casilla '{casilla}' no válida")
    return None

In [ ]:
def mover_a_casilla(casilla):
    """Mueve el brazo a la posición guardada para esa casilla."""
    punto = obtener_punto(casilla)
    if punto is None:
        return False
    base, elbow, hombro = punto
    base_corregido   = base + CORRECCION_BASE
    hombro_corregido = hombro + CORRECCION_HOMBRO
    elbow_corregido  = elbow + CORRECCION_CODO

    enviar(f"B{base_corregido},S{hombro_corregido},E{elbow_corregido}")
    return True

In [ ]:
def ir_home_libre():
    """Home seguro con pinza abierta."""
    print("  -> HOME libre (pinza abierta)")
    enviar("HOME")

def ir_home_carga():
    """Home para transportar piezas (pinza cerrada)."""
    print("  -> HOME carga (pinza cerrada)")
    # Si tu firmware requiere un comando especial, agrégalo aquí.
    # Por ahora no enviamos nada porque ya se controla con enviar(191/192)
    pass

In [ ]:
def enviar_antichoque(casilla):
    """Envía comando de antichoque: 195 para filas 1-4, 196 para filas 5-8."""
    fila = int(casilla[1])   # ej: "A3" → 3
    if fila <= 4:
        enviar(195)   # antichoque hombro
    else:
        enviar(196)   # antichoque codo

In [ ]:
def mover_pieza(origen, destino):
    """Secuencia completa para mover una pieza desde 'origen' hasta 'destino'."""
    print(f"\n=== MOVIMIENTO: {origen.upper()} -> {destino.upper()} ===")

    print("[1] Abriendo pinza")
    # La pinza se abre con el comando 192 (según tu firmware). 
    # Ajusta si es necesario.
    enviar(192)   # abre pinza

    print(f"[2] Yendo a origen: {origen.upper()}")
    if not mover_a_casilla(origen):
        return

    print("[3] Agarrando pieza")
    enviar(191)   # cierra pinza

    print("[4] Antichoque origen")
    enviar_antichoque(origen)

    print("[5] HOME carga")
    ir_home_carga()

    print(f"[6] Yendo a destino: {destino.upper()}")
    if destino.upper() == "194":
        # Comando especial para "tirar" pieza fuera del tablero
        enviar(194)
    else:
        if not mover_a_casilla(destino):
            ir_home_libre()
            return
        print("[7] Soltando pieza")
        enviar(192)   # abre pinza

        print("[8] Antichoque destino")
        enviar_antichoque(destino)

    print("[9] HOME libre")
    ir_home_libre()

    print("=== MOVIMIENTO COMPLETADO ===")

In [ ]:
print("\nBrazo robótico de ajedrez listo.")
print(f"Leyendo puntos desde: {ARCHIVO_PUNTOS}")
print("Comandos:")
print("  - 'a1,b3'          mueve pieza de a1 a b3")
print("  - 'a1,194'         captura pieza fuera del tablero")
print("  - 'home'           regresa a home libre")
print("  - 'puntos'         muestra las casillas cargadas")
print()

while True:
    entrada = input("Comando: ").strip().lower()
    if entrada == "":
        continue
    if entrada == "home":
        ir_home_libre()
        continue
    if entrada == "puntos":
        p = cargar_puntos()
        print(f"Casillas cargadas ({len(p)}):", list(p.keys()))
        continue
    if "," in entrada:
        partes = entrada.split(",")
        if len(partes) == 2:
            mover_pieza(partes[0].strip(), partes[1].strip())
        else:
            print("  Formato incorrecto. Usa: a1,b3")
    else:
        print("  Formato incorrecto. Usa: a1,b3 o a1,194")